In [2]:
# GloVe smoke test: pair scoring, paper loss, gradients, co-occurrences, and training.
from pathlib import Path
import sys

import torch

working_directory = Path.cwd()
project_candidates = [
    working_directory,
    *working_directory.parents,
    working_directory / "deep-learning-papers-pytorch",
]
project_root = next(
    (
        candidate
        for candidate in project_candidates
        if (candidate / "implementations/models/glove.py").is_file()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Could not locate the deep-learning-papers-pytorch project root")
sys.path.insert(0, str(project_root))

from implementations.models.glove import GloveModel, build_cooccurrence_matrix

torch.manual_seed(0)
vocab_size = 5
embedding_dim = 8
model = GloveModel(vocab_size=vocab_size, embedding_dim=embedding_dim)

# The model predicts dot(w_i, w_tilde_j) + b_i + b_tilde_j for observed pairs.
target_ids = torch.tensor([1, 1, 2, 3])
context_ids = torch.tensor([2, 3, 3, 1])
counts = torch.tensor([1.0, 0.5, 1.0, 0.5])
predictions = model(target_ids, context_ids)
assert predictions.shape == counts.shape

# Check the weighted squared-log-count objective and that gradients propagate.
loss = model.loss(target_ids, context_ids, counts)
assert loss.ndim == 0 and torch.isfinite(loss)
loss.backward()
assert all(
    parameter.grad is not None and torch.isfinite(parameter.grad).all()
    for parameter in model.parameters()
)

# A three-token sentence gives inverse-distance weights of 1 and 1/2.
cooccurrence = build_cooccurrence_matrix([[1, 2, 3]], vocab_size, window_size=2)
expected = torch.tensor(
    [
        [0.0, 0.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.5, 0.0],
        [0.0, 1.0, 0.0, 1.0, 0.0],
        [0.0, 0.5, 1.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 0.0],
    ]
)
assert torch.allclose(cooccurrence.to_dense(), expected)

# Run two AdaGrad epochs and verify finite objective history and vector shape.
history = model.fit(cooccurrence, epochs=2, shuffle=False)
embeddings = model.get_embeddings()
assert len(history) == 2 and torch.isfinite(torch.tensor(history)).all()
assert embeddings.shape == (vocab_size, embedding_dim)

print(f"Forward output shape: {tuple(predictions.shape)}")
print(f"Initial batch loss: {loss.item():.4f}")
print(f"Training objectives: {[round(value, 4) for value in history]}")
print(f"Combined embedding shape: {tuple(embeddings.shape)}")

c:\Users\Dell\Desktop\practice\deep-learning-papers-pytorch\implementations\models\glove.py:76: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\Context.cpp:855.)
  return torch.sparse_coo_tensor(


Forward output shape: (4,)
Initial batch loss: 0.0179
Training objectives: [0.0182, 0.0135]
Combined embedding shape: (5, 8)
